# Iterative experiment results (`runs/iter*`)

Plots for everything the VM session produced on the 8-wire low-data task
(w128d4 Adam lr 3e-3, `train_frac=0.5`, wire 0 unless stated). Sections map
to experiment families: noise dose-response, transient vs persist,
init scale, AdamW decay, LR/optimizer, Adam-eps spike test, 50k-step long
horizon (incl. the VM session's `noise_scale="rms"` variant, `_wnm` runs),
train_frac sweep, robustness (pool/circuit seeds, multi-wire), model-size
scaling, and a ranked summary. Thin markers = individual seeds; lines =
seed means. Chance (ln 2) dotted on loss panels.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np

from train import load_run

W = 0  # target wire
CHANCE = float(np.log(2))
ANY = ...

RUNS, _seen = [], set()
for p in sorted(glob.glob("runs/iter/*.npz")) + sorted(glob.glob("runs/iter_params/*.npz")):
    base = p.split("/")[-1]
    if base in _seen:
        continue  # iter_params duplicates some iter configs bit-for-bit
    _seen.add(base)
    cfg, d = load_run(p)
    cfg["ow"] = tuple(cfg["output_wires"] or ())
    RUNS.append((cfg, d))
print(f"{len(RUNS)} runs loaded")

BASE = dict(width=128, mlp_depth=4, lr=3e-3, steps=10_000, optimizer="adam",
            weight_decay=0.0, weight_noise=0.0, noise_mode="transient",
            noise_scale="init", init_scale=1.0, adam_eps=1e-8,
            train_frac=0.5, pool_seed=0, circuit_seed=2, ow=(0,))


def sel(**over):
    """Runs matching BASE with overrides; ANY (Ellipsis) matches anything."""
    want = {**BASE, **over}
    return [(c, d) for c, d in RUNS
            if all(v is ANY or c.get(k) == v for k, v in want.items())]


def group(runs, field):
    g = {}
    for c, d in runs:
        g.setdefault(c[field], []).append(d)
    return dict(sorted(g.items()))


def final(d, key="per_out_acc_ho"):
    a = d[key]
    return a[-1] if a.ndim == 1 else a[-1, W]


def _fmt(v):
    return v if isinstance(v, str) else f"{v:g}"


def traj(ax, runs, field, key="per_out_loss_ho", clip=1e-6, cmap=None):
    gs = group(runs, field)
    cols = (cmap or plt.cm.viridis)(np.linspace(0, 0.85, max(len(gs), 2)))
    for (val, ds), col in zip(gs.items(), cols):
        for i, d in enumerate(ds):
            m = d["eval_steps"] > 0
            a = d[key]
            y = a[m] if a.ndim == 1 else np.maximum(a[m][:, W], clip)
            ax.plot(d["eval_steps"][m], y, color=col, lw=1.1,
                    alpha=0.6 if len(ds) > 1 else 1.0,
                    label=f"{field}={_fmt(val)}" if i == 0 else None)
    if "loss" in key:
        ax.axhline(CHANCE, color="gray", ls=":", lw=0.8)
    ax.set(xscale="log", yscale="log", xlabel="step", ylabel=key)
    ax.legend(fontsize=7)


def finals(ax, runs, field, key="per_out_acc_ho", color="C0", label=None, ls="-"):
    gs = group(runs, field)
    for x, ds in gs.items():
        ax.plot([x] * len(ds), [final(d, key) for d in ds], "o",
                color=color, ms=4, alpha=0.45)
    ax.plot(list(gs), [np.mean([final(d, key) for d in ds]) for ds in gs.values()],
            ls, color=color, lw=1.6, marker="o", ms=5, label=label)
    ax.set(xlabel=field, ylabel=f"final {key}")
    if label:
        ax.legend(fontsize=8)

## Noise dose-response (transient, init-relative)

Final held-out accuracy vs eta, plus held-out BCE and weight-norm
trajectories at selected doses.

In [ ]:
noise = sel(weight_noise=ANY)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
finals(axes[0], noise, "weight_noise")
axes[0].axhline(1.0, color="gray", ls=":", lw=0.8)
axes[0].set(xlabel="noise eta (fraction of init std)", ylim=(0.45, 1.02))
pick = [r for r in noise if r[0]["weight_noise"] in (0, 0.15, 0.5, 1)]
traj(axes[1], pick, "weight_noise")
axes[1].set(ylabel="held-out BCE (wire 0)")
traj(axes[2], pick, "weight_noise", key="param_norm")
axes[2].set(ylabel="param L2 norm", yscale="linear")
plt.tight_layout()

## Transient vs persistent noise

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for mode, col in (("transient", "C0"), ("persist", "C3")):
    rs = [r for r in sel(weight_noise=ANY, noise_mode=mode)
          if r[0]["weight_noise"] in (0, 0.1, 0.3)]
    finals(ax, rs, "weight_noise", color=col, label=mode)
ax.set(ylim=(0.45, 1.02))
plt.tight_layout()

## Init scale (Omnigrok lever)

Clean init-scale sweep, the wn=0.5 combos, and clean trajectories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
finals(axes[0], sel(init_scale=ANY), "init_scale", label="clean (wn=0)")
finals(axes[0], sel(init_scale=ANY, weight_noise=0.5), "init_scale",
       color="C1", label="wn=0.5")
axes[0].set(ylim=(0.45, 1.02))
traj(axes[1], sel(init_scale=ANY), "init_scale")
plt.tight_layout()

## AdamW weight decay (10k steps)

In [ ]:
wd_runs = sel(optimizer="adamw", weight_decay=ANY) + sel()  # + adam baseline
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
finals(axes[0], wd_runs, "weight_decay")
axes[0].set(ylim=(0.45, 1.02))
traj(axes[1], wd_runs, "weight_decay")
plt.tight_layout()

## Learning rate and optimizer (Adam LR sweep; SGD arms)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for wn, col in ((0.0, "C0"), (0.5, "C1")):
    finals(axes[0], sel(lr=ANY, weight_noise=wn), "lr",
           color=col, label=f"adam wn={wn:g}")
axes[0].set(xscale="log", ylim=(0.45, 1.02), title="Adam")
for wn, col in ((0.0, "C0"), (0.5, "C1"), (1.0, "C2")):
    finals(axes[1], sel(optimizer="sgd", lr=ANY, weight_noise=wn), "lr",
           color=col, label=f"sgd wn={wn:g}")
axes[1].set(xscale="log", ylim=(0.45, 1.02), title="SGD (momentum 0.9)")
plt.tight_layout()

## Adam epsilon: are the spikes load-bearing?

Left: final held-out acc vs eps across three settings. Middle/right:
train-pool (spike view) and held-out BCE at 50k steps, wn=0.5.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
finals(axes[0], sel(adam_eps=ANY), "adam_eps", label="clean 10k")
finals(axes[0], sel(adam_eps=ANY, weight_noise=1.0, train_frac=0.3),
       "adam_eps", color="C1", label="wn=1 tf=0.3")
finals(axes[0], sel(adam_eps=ANY, weight_noise=0.5, steps=50_000),
       "adam_eps", color="C2", label="wn=0.5 50k")
axes[0].set(xscale="log", ylim=(0.45, 1.02))
e50 = sel(weight_noise=0.5, steps=50_000, adam_eps=ANY)
traj(axes[1], e50, "adam_eps", key="per_out_loss_tr", clip=1e-8)
axes[1].set(ylabel="train-pool BCE")
traj(axes[2], e50, "adam_eps")
axes[2].set(ylabel="held-out BCE")
plt.tight_layout()

## Long horizon (50k steps, w128d4)

All 50k arms, including the VM session's RMS-anchored noise
(`noise_scale="rms"`, `_wnm` tag) and adamw+noise.

In [ ]:
arms = {
    "wn0.3": sel(weight_noise=0.3, steps=50_000),
    "wn0.5": sel(weight_noise=0.5, steps=50_000),
    "wn1": sel(weight_noise=1.0, steps=50_000),
    "wn0.5 rms-scaled": sel(weight_noise=0.5, steps=50_000, noise_scale="rms"),
    "adamw0.01 + wn0.5": sel(optimizer="adamw", weight_decay=0.01,
                             weight_noise=0.5, steps=50_000),
}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
cols = plt.cm.tab10(np.arange(len(arms)))
for (lab, rs), col in zip(arms.items(), cols):
    for i, (c, d) in enumerate(rs):
        m = d["eval_steps"] > 0
        axes[0].plot(d["eval_steps"][m],
                     np.maximum(d["per_out_loss_ho"][m][:, W], 1e-6),
                     color=col, lw=1.1, alpha=0.7, label=lab if i == 0 else None)
        axes[1].plot(d["eval_steps"][m], d["per_out_acc_ho"][m][:, W],
                     color=col, lw=1.1, alpha=0.7)
axes[0].axhline(CHANCE, color="gray", ls=":", lw=0.8)
axes[0].set(xscale="log", yscale="log", xlabel="step", ylabel="held-out BCE")
axes[0].legend(fontsize=7)
axes[1].axhline(1.0, color="gray", ls=":", lw=0.8)
axes[1].set(xscale="log", xlabel="step", ylabel="held-out acc")
plt.tight_layout()

## Train-frac sweep

Shaded region: below train_frac ~0.2-0.25 the function is no longer
identifiable in principle (consistent impostor circuits exist), so a gap to
1.0 there is partly the task's fault, not the learner's.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.4))
for wn, col in ((0.0, "C0"), (0.5, "C1"), (1.0, "C2")):
    finals(ax, sel(train_frac=ANY, weight_noise=wn), "train_frac",
           color=col, label=f"wn={wn:g}")
ax.axvspan(0.15, 0.25, color="gray", alpha=0.15)
ax.axhline(1.0, color="gray", ls=":", lw=0.8)
ax.set(ylim=(0.45, 1.02))
plt.tight_layout()

## Robustness: pool seed, circuit seed, multi-wire

Circuit seeds are different tasks (different functions), so read that panel
as method robustness, not seed noise. Multi-wire = all 8 outputs trained
jointly; wire-0 accuracy shown for comparability.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for wn, col in ((0.0, "C0"), (0.5, "C1")):
    finals(axes[0], sel(pool_seed=ANY, weight_noise=wn), "pool_seed",
           color=col, label=f"wn={wn:g}")
    finals(axes[1], sel(circuit_seed=ANY, weight_noise=wn), "circuit_seed",
           color=col, label=f"wn={wn:g}")
for ax in axes[:2]:
    ax.set(ylim=(0.45, 1.02))
for wn, col, off in ((0.0, "C0", -0.08), (0.5, "C1", 0.08)):
    for x, ow in ((0, (0,)), (1, ())):
        vals = [final(d) for c, d in sel(ow=ow, weight_noise=wn)]
        axes[2].plot([x + off] * len(vals), vals, "o", color=col, ms=6,
                     alpha=0.6, label=f"wn={wn:g}" if x == 0 else None)
axes[2].set(xticks=[0, 1], xticklabels=["solo (wire 0)", "all 8 wires"],
            xlim=(-0.5, 1.5), ylim=(0.45, 1.02),
            ylabel="final held-out acc (wire 0)")
axes[2].legend(fontsize=8)
plt.tight_layout()

## Model-size scaling

10k-step clean vs wn=0.5 across shapes at both LRs; stars are the 30k-step
big-shape arms (various wn/eps).

In [ ]:
SHAPES = [(32, 2), (64, 3), (128, 4), (180, 5), (256, 6), (360, 7), (512, 8)]


def n_params(w, d, hr=4):
    h = hr * w
    return 8 * w + d * (w + w * h + h * w) + w + w * 8


fig, ax = plt.subplots(figsize=(7.5, 4.6))
for lr, ls in ((1e-3, "--"), (3e-3, "-")):
    for wn, col in ((0.0, "C0"), (0.5, "C1")):
        xs, ys = [], []
        for w, dep in SHAPES:
            rs = sel(width=w, mlp_depth=dep, lr=lr, weight_noise=wn)
            if rs:
                xs.append(n_params(w, dep))
                ys.append(np.mean([final(d) for _, d in rs]))
        ax.plot(xs, ys, ls, color=col, marker="o", ms=4,
                label=f"lr={lr:g} wn={wn:g} (10k)")
rs30 = sel(width=ANY, mlp_depth=ANY, lr=ANY, steps=30_000,
           weight_noise=ANY, adam_eps=ANY, optimizer=ANY, weight_decay=ANY)
ax.plot([n_params(c["width"], c["mlp_depth"]) for c, d in rs30],
        [final(d) for c, d in rs30], "*", color="C3", ms=10, ls="none",
        label="30k arms (wn 0.3-0.5)")
ax.axhline(1.0, color="gray", ls=":", lw=0.8)
ax.set(xscale="log", xlabel="params N", ylabel="final held-out acc",
       ylim=(0.45, 1.05))
ax.legend(fontsize=7)
plt.tight_layout()

## Ranked summary — every config, best first (mean over seeds)

In [ ]:
import collections

groups = collections.defaultdict(list)
for c, d in RUNS:
    key = tuple((k, c.get(k)) for k in BASE)
    groups[key].append(final(d))


def describe(key):
    parts = [f"{k}={_fmt(v) if not isinstance(v, tuple) else v}"
             for k, v in key if BASE.get(k) != v]
    return " ".join(parts) or "baseline (w128d4 adam lr3e-3 tf0.5)"


rows = sorted(groups.items(), key=lambda kv: -np.mean(kv[1]))
print(f"{'ho acc':>7s} {'n':>2s}  config (diffs from baseline)")
for key, accs in rows[:25]:
    print(f"{np.mean(accs):>7.3f} {len(accs):>2d}  {describe(key)}")
print("   ...")
for key, accs in rows[-5:]:
    print(f"{np.mean(accs):>7.3f} {len(accs):>2d}  {describe(key)}")